In [1]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from dotenv.ipython  import load_dotenv


In [2]:
load_dotenv(override=True)

True

In [3]:
llm = ChatOpenAI(model="gpt-4o", temperature=0)

In [4]:
agent = create_agent(model=llm,system_prompt="you are a helpful assistant ")

In [5]:
resp = agent.invoke(
    {"messages": [{"role": "user", "content": "je m'appelle Mohamed"}]},
    context={"env": "test"}
)


AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-proj-********************************************************************************************************************************************************erEA. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}

In [ ]:
print(resp['messages'][-1].content)

Bonjour Mohamed ! Comment puis-je vous aider aujourd'hui ?


In [ ]:
resp = agent.invoke(
    {"messages": [{"role": "user", "content": "Comment je m'appelle"}]},
    context={"env": "test"}
)


In [ ]:
print(resp['messages'][-1].content)

Je suis désolé, mais je ne connais pas votre nom. Si vous me l'avez déjà dit dans une conversation précédente, je ne peux pas m'en souvenir. Vous pouvez me le dire si vous le souhaitez !


In [ ]:
from langchain.agents.middleware import ModelRequest,ModelResponse
from langchain.messages import HumanMessage,SystemMessage,AIMessage

In [ ]:
basic_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
advanced_llm = ChatOpenAI(model="gpt-4o", temperature=0)


In [ ]:
@wrap_model_call
def dynamic_model_selection(request: ModelRequest,handler)-> ModelResponse:
    env = request.runtime.context.get("env","test")
    if env == "test":
        model = basic_llm
        print("basic_llm selected")
    else:
        model = advanced_llm
        print("advanced_llm selected")

    return handler(request.override(model=model))

In [ ]:
agent2 = create_agent(
    model=basic_llm,
    tools=[],
    middleware=[dynamic_model_selection],)

In [ ]:
resp = agent2.invoke(
    input={
        "messages":[HumanMessage("C'est quoi un agent AI")]
    },
    context={"env": "test"}
)

basic_llm selected


In [ ]:
from IPython.display import Markdown

In [ ]:
print(display(Markdown(resp['messages'][-1].content)))

Un agent AI (intelligence artificielle) est un système ou un programme informatique capable d'effectuer des tâches qui nécessitent normalement une intelligence humaine. Ces agents peuvent percevoir leur environnement, prendre des décisions, apprendre de nouvelles informations et interagir avec les utilisateurs ou d'autres systèmes. 

Il existe différents types d'agents AI, notamment :

1. **Agents réactifs** : Ils réagissent à des stimuli dans leur environnement sans mémoire ou capacité d'apprentissage. Par exemple, un thermostat qui ajuste la température en fonction de la chaleur ambiante.

2. **Agents basés sur des modèles** : Ils utilisent des modèles internes pour représenter leur environnement et prendre des décisions basées sur ces modèles.

3. **Agents d'apprentissage** : Ces agents peuvent apprendre de leurs expériences et améliorer leurs performances au fil du temps, comme les systèmes de recommandation ou les assistants virtuels.

4. **Agents autonomes** : Ils peuvent fonctionner de manière indépendante pour accomplir des tâches spécifiques, comme les drones ou les robots.

Les agents AI sont utilisés dans divers domaines, y compris la santé, la finance, le service client, et bien d'autres, pour automatiser des processus, analyser des données et améliorer l'efficacité.

None


In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

In [ ]:
memory = InMemorySaver()
agent = create_agent(
    model="openai:gpt-4o",
    system_prompt="you are a helpful assistant ",
    checkpointer=memory)

In [ ]:
config= {"configurable":{"thread_id":1}}
resp = agent.invoke(input = {"messages":[HumanMessage("Je m'appelle hamid")]},config=config)


In [ ]:
print(resp['messages'][-1].content)

Bonjour Hamid ! Comment puis-je vous aider aujourd'hui ?


In [ ]:
resp = agent.invoke(input = {"messages":[HumanMessage("Comment je m'apelle ?")]},config=config)


In [ ]:
print(resp['messages'][-1].content)

Vous avez dit que vous vous appelez Hamid. C’est bien ça ?


In [ ]:
from langchain.tools import tool
from langchain.agents import create_agent

In [ ]:
@tool
def get_weather(city:str):
    """
    Get The weather of given city
    """
    print("Weather Tool Invoked")
    return{
        "city":city,
        "temperature":23,
        "humidity":78,
        "pressure":120
    }

In [ ]:
@tool
def get_employee_info(employee_name:str):
    """
    Get Information about a specific employee
    """
    print("Employee Info Tool Invoked")
    return{
        "name":employee_name,
        "salary":5000,
        "seniority":5
    }

In [ ]:
agent4 = create_agent(
    model="openai:gpt-4o",
    tools=[get_weather,get_employee_info],
    checkpointer=memory,
    system_prompt="you are a helpful assistant that can provide weather information and employee information based on the user's request"
)


In [ ]:
config = {"configurable":{"thread_id":1}}
resp = agent4.invoke(input={
    "messages":[HumanMessage("La météo à casablanca")]
},config=config)

Weather Tool Invoked


In [ ]:
print(resp['messages'][-1].content)

La météo à Casablanca indique une température de 23°C, avec une humidité de 78% et une pression atmosphérique de 120 hPa. Si vous avez besoin d'autres informations, n'hésitez pas à me le faire savoir !


In [ ]:
load_dotenv(override=True)

True

In [ ]:
from langchain_tavily import TavilySearch

In [ ]:
tavily = TavilySearch(max_results=10,search_depth="advanced")

In [ ]:
@tool
def search_web(query:str):
    """Search the web for a given query and return relevant results"""

    print(f"Search Web Tool Invoked with {query}")
    results = tavily.invoke(
        input={"query":query}
    )
    return results

In [ ]:
agent5 = create_agent(
    model="openai:gpt-4o",
    tools=[get_weather,get_employee_info,search_web],
    checkpointer=memory,
    system_prompt="you are a helpful assistant that can provide weather information and employee information based on the user's request"
)


In [6]:
resp = agent5.invoke(input={
    "messages":[HumanMessage("Je veux l'actualité sur le maroc")]
},config=config)

NameError: name 'agent5' is not defined

In [ ]:
print(resp['messages'][-1].content)

Selon les différentes sources web, la météo actuelle à Casablanca est plutôt ensoleillée avec des températures autour de 20°C pendant la journée. La nuit, les températures peuvent descendre jusqu'à 12°C. Il n'y a presque pas de précipitations prévues. Vous pouvez consulter des ressources comme [Météo et Radar](https://www.meteoetradar.com/meteo/casablanca/4162015) ou [MeteoArt.com](https://www.meteoart.com/africa/morocco/casablanca?page=today) pour des informations plus détaillées et mises à jour régulièrement.

Si vous avez besoin d'autres informations, n'hésitez pas à me le dire !


In [ ]:
from langchain_experimental.toosl import PythonREPLTool
repl_tool = PythonREPLTool(sanitize_input=False)

agent6 = create_agent(
    model="openai:gpt-4o",
    tools=[get_weather,get_employee_info,search_web,repl_tool],
    system_prompt="Générer et exécuter le code python en placant le code python et le résultat d'executionde ce code dans le fichier doc.txt"
)


ModuleNotFoundError: No module named 'langchain_experimental.toosl'

In [8]:
resp = agent6.invoke(input={
    "messages":[HumanMessage("Je veux trier les deux listes suivantes [5,2,9] et [3,8,1] et puis additionner les 2 listes ")]
},config=config)


NameError: name 'agent6' is not defined

In [ ]:
print(resp['messages'][-1].content)